# Clustering the sample space — the exploration that seeded the KDE defence

Before KDE became one of the paper's defences, it was tried here as a way to *score* samples: cluster the training samples, then measure how a new sample sits against the cluster centres with a Gaussian-kernel density estimate. This notebook is that exploration, in four parts: KMeans + cluster-KDE, the same restricted to **non-removable features** (the ones an attacker cannot take away), a DBSCAN sweep, and a final large KMeans run.

*Consolidated from `clustering.ipynb` (originally in `mal_adv4`). Repeated import and KDE-call cells from the original exploration were removed; the parameter values of every DBSCAN run are kept. Outputs cleared.*

## Step 1 — Setup: data and the trained detectors under study

In [ ]:

download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM_weightedLoss.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

In [ ]:
import gdown
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pickle
import random
import time

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from adverserial_attacks_functions import *


torch.manual_seed(0)



In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array).to(device)
print(len(removal_array))


In [ ]:

# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()
print('the proportion of malwares : ', n_mal/(n_mal+n_ben))

# Combine features and labels into datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 20000
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
del train_dataset, val_dataset, test_dataset, y_train, y_val, y_test


In [ ]:
model_DNN = MalwareDetectionModel().to(device)
# Load model parameters
model_DNN.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_rFGSM_weightedLoss = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM_weightedLoss.load_state_dict(torch.load('model_AT_rFGSM_weightedLoss.pth', map_location=torch.device(device)))

In [ ]:
for x,y in test_loader:
  print(x.shape)
  break

bens = x[y.squeeze()==0]
mals = x[y.squeeze()==1]
print(bens.shape)
print(mals.shape)

## Part 1 — KMeans clusters and KDE scoring

### Step 2 — Fit KMeans and count the clusters

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np
kmeans = KMeans(n_clusters=500, init='k-means++', n_init='auto')
cluster_labels = kmeans.fit_predict(bens)
print(kmeans.inertia_)
#silhouette_avg = silhouette_score(bens, cluster_labels)
#print(silhouette_avg)


In [ ]:
cluster_centers = torch.tensor(kmeans.cluster_centers_)
cluster_centers.shape

In [ ]:
kmeans.labels_

In [ ]:
# Count the number of samples in each cluster
cluster_counts = torch.tensor(np.bincount(cluster_labels))

# Print the number of samples in each cluster
#for cluster_id, count in enumerate(cluster_counts):
   # print(f"Cluster {cluster_id}: {count} samples")

In [ ]:
print(cluster_counts.shape)
print(cluster_counts)

In [ ]:
cluster_counts.sum()

### Step 3 — The Gaussian-kernel KDE over cluster centres

In [ ]:
import torch

# Define the multivariate Gaussian kernel function
def gaussian_kernel(u):
    """
    Compute the value of the Gaussian kernel function at point u.

    Parameters:
        u (torch.Tensor): Standardized distance between data points and evaluation points.

    Returns:
        torch.Tensor: Array of Gaussian kernel values.
    """
    d = u.shape[1]  # Number of dimensions (features) of the data
    # normalization constant for the Gaussian kernel in d dimensions
    #normalization = (2 * np.pi) ** (-d / 2)
    normalization = 10.  # normalization constant for the Gaussian kernel in d dimensions
    return normalization * torch.exp(-0.5 * torch.sum(u**2, dim=-1))

# Multidimensional KDE implementation
def KDE(x, data, cluster_counts, bandwidth):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    n = data.shape[0]  # Number of data points
    d = x.shape[1]  # Dimensionality of the data

    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate standardized distances for all data points
    u = (x[:, None, :] - data) / bandwidth_tensor
    #print(u.shape)
    # Compute Gaussian kernel contributions for all data points
    kernel_contributions = gaussian_kernel(u)
    #print(kernel_contributions.shape)
    # Sum contributions across all data points
    #kernel_contributions = torch.ones((40, 100))
    #print((kernel_contributions).sum())
    #print((kernel_contributions* cluster_counts[np.newaxis,:]).sum()*cluster_counts.shape/cluster_counts.sum())
    #estimate = torch.mean((kernel_contributions* cluster_counts[np.newaxis,:]), dim=1) * cluster_counts.shape[0] / cluster_counts.sum()
    estimate = torch.mean((kernel_contributions), dim=1)
    print(estimate.sum())
    # Normalize the density estimate by the number of points and the bandwidth raised to the dimensionality
    #estimate /= (bandwidth_tensor ** d)

    return estimate


def KDE_wheighted(x, data, cluster_counts, bandwidth):
    """
    Compute the kernel density estimate (KDE) for given data points.

    Parameters:
        x (torch.Tensor): Points at which to evaluate the KDE (shape: [num_samples, num_dimensions]).
        data (torch.Tensor): Data points used to estimate the density (shape: [num_data_points, num_dimensions]).
        bandwidth (float): Bandwidth parameter for the KDE.

    Returns:
        torch.Tensor: Density estimate at each point in x (shape: [num_samples,]).
    """
    n = data.shape[0]  # Number of data points
    d = x.shape[1]  # Dimensionality of the data

    # Convert bandwidth to tensor
    bandwidth_tensor = torch.tensor(bandwidth)

    # Calculate standardized distances for all data points
    u = (x[:, None, :] - data) / bandwidth_tensor

    # Compute Gaussian kernel contributions for all data points
    kernel_contributions = gaussian_kernel(u)
    estimate = torch.mean((kernel_contributions* cluster_counts[np.newaxis,:]), dim=1) * cluster_counts.shape[0] / cluster_counts.sum()
    print(estimate.sum())


    return estimate

In [ ]:
cluster_counts.shape

In [ ]:
KDE(mals[:50], cluster_centers,cluster_counts, 0.6)

In [ ]:
KDE(bens[:50], cluster_centers,cluster_counts, 0.6)

## Part 2 — Only non-removable features

The original notebook's own heading is kept:

# **Just non_removable features**

In [ ]:
removal_array.sum()

In [ ]:
non_removal_array = torch.tensor((1 - removal_array), dtype=torch.bool)
non_removal_array.sum()

In [ ]:
non_removal_array

In [ ]:
x2 = x[:,non_removal_array]
print(x2.shape)

bens2 = x2[y.squeeze()==0]
mals2 = x2[y.squeeze()==1]
print(bens2.shape)
print(mals2.shape)

### Step 4 — Re-cluster the reduced samples and score again (now also with the weighted KDE)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np
kmeans = KMeans(n_clusters=500, init='k-means++', n_init='auto')
cluster_labels = kmeans.fit_predict(bens2)
print(kmeans.inertia_)
#silhouette_avg = silhouette_score(bens, cluster_labels)
#print(silhouette_avg)


In [ ]:
# Count the number of samples in each cluster
cluster_counts = torch.tensor(np.bincount(cluster_labels))

print(cluster_counts.shape)
print(cluster_counts)

In [ ]:
KDE(mals2[:50], cluster_centers,cluster_counts, 0.6)

In [ ]:
KDE(bens2[:50], cluster_centers,cluster_counts, 0.6)

In [ ]:
KDE_wheighted(mals2[:50], cluster_centers,cluster_counts, 0.6)

In [ ]:
KDE_wheighted(bens2[:50], cluster_centers,cluster_counts, 0.6)

## Part 3 — DBSCAN sweep

Several (eps, min_samples) combinations, checking how many benign and malicious samples fall into the outlier set (label −1) and into each cluster. The repeated DBSCAN import cells of the original are collapsed into one.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import numpy as np

# Assuming 'high_dim_data' is your high-dimensional dataset, stored in a numpy array or pandas DataFrame
# Example: high_dim_data = ...

# Define the parameter grid
eps_values = [3.1]      # Adjust range and step size as needed
min_samples_values = range(2, 61)              # Adjust range as needed

best_params = None
best_score = -1

# Iterate over parameter combinations
for eps in eps_values:
    for min_samples in min_samples_values:


        print('*******************************')
        print('eps,min_samples : ',eps,min_samples)

        # Compute silhouette score
        #silhouette_avg = silhouette_score(bens, cluster_labels)
        #print('silhouette_avg',silhouette_avg)

        dbscan = DBSCAN(eps=eps, min_samples=min_samples)

        # Fit DBSCAN model to the data
        cluster_labels = dbscan.fit_predict(x)
        n_clusters_ = len(set(dbscan.labels_))
        print('n_clusters_ : ', n_clusters_)

        outliers_mask = (dbscan.labels_ == -1)

        print('number of outliers : ',outliers_mask.sum())
        print('number of malwares in outliers : ',y[outliers_mask].sum())

        for i in range(10):
          print('cluster : ',i)
          mask = dbscan.labels_ == i
          print('number of samples in cluster : ',mask.sum())
          print('number of malwares in cluster : ',y[mask].sum())

In [ ]:
eps = 4.1
min_samples = 53

# Initialize DBSCAN model with current parameter combination
dbscan = DBSCAN(eps=eps, min_samples=min_samples)

# Fit DBSCAN model to the data
cluster_labels = dbscan.fit_predict(x)

outliers_mask = (dbscan.labels_ == -1)
print('eps,min_samples : ',eps,min_samples)
n_clusters_ = len(set(dbscan.labels_))
print('n_clusters_ : ', n_clusters_)
print('number of outliers : ',(outliers_mask).sum())


In [ ]:
# Identify outliers (data points with label -1)
mask = dbscan.labels_ == 0
print(mask.sum())
print(y[mask].sum())

In [ ]:
len(bens)

In [ ]:
eps = 3.1
min_samples = 36

# Initialize DBSCAN model with current parameter combination
dbscan = DBSCAN(eps=eps, min_samples=min_samples)

# Fit DBSCAN model to the data
cluster_labels = dbscan.fit_predict(x)



In [ ]:
mask = dbscan.labels_ == -1
print(mask.sum())
print(y[mask].sum())

In [ ]:
mask = dbscan.labels_ == 0
print(mask.sum())
print(y[mask].sum())

In [ ]:
mask = dbscan.labels_ == 1
print(mask.sum())
print(y[mask].sum())

In [ ]:
mask = dbscan.labels_ == 2
print(mask.sum())
print(y[mask].sum())

In [ ]:
mask = dbscan.labels_ == 3
print(mask.sum())
print(y[mask].sum())

In [ ]:
eps = 4.1
min_samples = 30

# Initialize DBSCAN model with current parameter combination
dbscan = DBSCAN(eps=eps, min_samples=min_samples)

# Fit DBSCAN model to the data
cluster_labels = dbscan.fit_predict(x)

outliers_mask = (dbscan.labels_ == -1)
print('eps,min_samples : ',eps,min_samples)
n_clusters_ = len(set(dbscan.labels_))
print('n_clusters_ : ', n_clusters_)
print('number of outliers : ',(outliers_mask).sum())


In [ ]:
# Identify number of benigns (data points with label -1)
mask = dbscan.labels_ == 0
print(mask.sum())
print(y[mask].sum())

In [ ]:
# Identify outliers (data points with label -1)
mask = dbscan.labels_ == 1
print(mask.sum())
print(y[mask].sum())

In [ ]:
dbscan = DBSCAN(eps=4.1, min_samples=31)

# Fit DBSCAN model to the data
cluster_labels = dbscan.fit_predict(bens)

outliers_mask = (dbscan.labels_ == -1)

n_clusters_ = len(set(dbscan.labels_))
print('n_clusters_ : ', n_clusters_)
print('number of outliers : ',(outliers_mask).sum())
# Compute silhouette score
silhouette_avg = silhouette_score(bens, cluster_labels)
print('silhouette_avg',silhouette_avg)

In [ ]:
mask = (dbscan.labels_ == 0)
print((mask).sum())
mask = (dbscan.labels_ == -1)
print((mask).sum())

In [ ]:
# Identify outliers (data points with label -1)
outliers_mask = dbscan.labels_ == -1
outliers = bens[outliers_mask]

# Print or further analyze the outliers
print("Outliers:", outliers.shape)


In [ ]:
# Forward pass to get logits for benign samples
with torch.no_grad():  # No need for gradients
    outputs = model_AT_rFGSM(outliers.to(torch.float32))

# Calculate softmax probabilities
probabilities = torch.softmax(outputs, dim=1)

#number of wrong prediction for outliers
_, predicted = torch.topk(outputs, k=1)
predicted.sum()

## Part 4 — Final run: KMeans with 750 clusters

### Step 5 — Choose k (elbow + silhouette) and fit

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# Assuming 'binary_data' is your binary vectors data, stored in a numpy array or pandas DataFrame
binary_data = bens

# Calculate WCSS and silhouette score for different number of clusters
max_clusters = 100
wcss = []
silhouette_scores = []
for i in range(2, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, init='k-means++', n_init='auto', random_state=42)
    cluster_labels = kmeans.fit_predict(binary_data)
    wcss.append(kmeans.inertia_)
    silhouette_avg = silhouette_score(binary_data, cluster_labels)
    silhouette_scores.append(silhouette_avg)

    print("****************")
    print('number of clusters : ',i)
    print('inertia : ',kmeans.inertia_)
    print('silhouette_avg : ',silhouette_avg)




In [ ]:
# Plot the Elbow Method curve and Silhouette Score on the same plot
fig, ax1 = plt.subplots()

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters')
ax1.set_ylabel('WCSS', color=color)
ax1.plot(range(2, 87 + 1), wcss, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(range(2, 86 + 1), silhouette_scores, color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.title('Elbow Method and Silhouette Score')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters= 750, init='k-means++', n_init='auto', random_state=42)
cluster_labels = kmeans.fit_predict(binary_data)
print(kmeans.inertia_)
silhouette_avg = silhouette_score(binary_data, cluster_labels)
print(silhouette_avg)

In [ ]:
cluster_centers[0][:200]

### Step 6 — KDE scores on the final centres

In [ ]:
KDE(mals[:100], cluster_centers, 0.6)

In [ ]:
KDE(bens[:100], cluster_centers, 0.6)